In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import pickle
import optuna

In [2]:
data = pd.read_excel('07_ODU2_5min_for_energy_prediction_model.xlsx')

In [3]:
data.shape

(1735, 15)

In [4]:
data.columns

Index(['ST', 'IAT', 'OAT', 'Delta_IAT_OAT', 'Delta_ST_IAT', 'Delta_ST_OAT',
       'High_Pressure', 'Day_sin', 'Day_cos', 'Op_Mode_0', 'Op_Mode_1',
       'Year_sin', 'Year_cos', 'OTW', 'Energy'],
      dtype='object')

In [5]:
data.head()

,ST,IAT,OAT,Delta_IAT_OAT,Delta_ST_IAT,Delta_ST_OAT,High_Pressure,Day_sin,Day_cos,Op_Mode_0,Op_Mode_1,Year_sin,Year_cos,OTW,Energy
0,24.664311,24.853527,-1.059259,25.912786,-0.189216,25.723570,30.103704,0.779884,-0.625923,0,1,-0.179562,0.983747,-1.863094,13381.925926
1,24.664311,28.440094,-0.950000,29.390094,-3.775783,25.614311,30.733333,0.766044,-0.642788,0,1,-0.179503,0.983757,-1.892156,12357.900000
2,24.664311,29.182667,-0.910526,30.093193,-4.518356,25.574837,30.089474,0.751840,-0.659346,0,1,-0.179444,0.983768,-1.864571,9970.947368
3,24.664311,28.434894,-0.800000,29.234894,-3.770583,25.464311,28.137500,0.737277,-0.675590,0,1,-0.179385,0.983779,-1.823869,6738.000000
4,24.664311,28.165769,-0.800000,28.965769,-3.501458,25.464311,29.806250,0.722364,-0.691513,0,1,-0.179327,0.983790,-1.784245,8008.437500


In [6]:
# The normalizing the inputs

def normalize_inputs(data):
    ST_min, ST_max = 18, 31
    IAT_min, IAT_max = 15, 35
    OAT_min, OAT_max = -20, 20
    OTW_min, OTW_max = -15, 10
    High_Pressure_min, High_Pressure_max = 5, 35
    Energy_min, Energy_max = 0, 18000
    
    data['ST'] = (data['ST'] - ST_min) / (ST_max - ST_min)
    data['IAT'] = (data['IAT'] - IAT_min) / (IAT_max - IAT_min)
    data['OAT'] = (data['OAT'] - OAT_min) / (OAT_max - OAT_min)
    data['OTW'] = (data['OTW'] - OTW_min) / (OTW_max - OTW_min)
    data['High_Pressure'] = (data['High_Pressure'] - High_Pressure_min) / (High_Pressure_max - High_Pressure_min)
    data['Energy'] = (data['Energy'] - Energy_min) / (Energy_max - Energy_min)
    
    
    return data

# Denormalizing the predicted outputs

def inverse_normalization_output(outputs):
    Energy_min, Energy_max = 0, 18000
    outputs['Energy'] = outputs['Energy'] * (Energy_max - Energy_min) + Energy_min
    
    return outputs

In [7]:
#Normalizing outputs
data = normalize_inputs(data)

In [8]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
	n_vars = 1 if type(data) is list else data.shape[1]
	df = pd.DataFrame(data)
	cols, names = list(), list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = pd.concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg

In [9]:
values = data.values.astype('float32')
reframed = series_to_supervised(values,2, 1)
reframed = reframed.values

In [10]:
features = reframed[:, :-1]
targets = reframed[:, -1]

In [11]:
train_x, test_x, train_y, test_y = train_test_split(features, targets, test_size=0.2, shuffle=True, random_state=123)

In [12]:
import time
start = time.time()

In [13]:
def objective(trial):

    param = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 3000, step=20),
        'n_estimators': trial.suggest_categorical('n_estimators', [10, 30, 50, 100, 200, 500]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.001, 0.01, 0.1, 0.3, 0.008,0.009]),
        'max_depth': trial.suggest_categorical('max_depth', [3,5,7,9,11,13,15,17,19]),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda':trial.suggest_loguniform('reg_lambda', 0.5, 1.0),
        
    }
    
    cv_score_lr = cross_val_score(LGBMRegressor(**param), train_x, train_y, cv=5,scoring='r2')

    mean_r2 = round(sum(cv_score_lr)/len(cv_score_lr), 3)
    

    return mean_r2

In [14]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)
print('Number of finished trials:', len(study.trials))

[I 2023-02-02 08:53:43,950] A new study created in memory with name: no-name-17e4626e-2e00-4a0e-8b3e-e114115c96e2
[I 2023-02-02 08:53:45,046] Trial 0 finished with value: 0.716 and parameters: {'num_leaves': 860, 'n_estimators': 100, 'learning_rate': 0.009, 'max_depth': 15, 'reg_alpha': 0.0899494265461803, 'reg_lambda': 0.608385994400347}. Best is trial 0 with value: 0.716.
[I 2023-02-02 08:53:45,199] Trial 1 finished with value: 0.435 and parameters: {'num_leaves': 680, 'n_estimators': 50, 'learning_rate': 0.008, 'max_depth': 3, 'reg_alpha': 0.162404389061615, 'reg_lambda': 0.6765764223001542}. Best is trial 0 with value: 0.716.
[I 2023-02-02 08:53:45,534] Trial 2 finished with value: 0.85 and parameters: {'num_leaves': 2280, 'n_estimators': 200, 'learning_rate': 0.3, 'max_depth': 7, 'reg_alpha': 3.324104049824672, 'reg_lambda': 0.6741526410714997}. Best is trial 2 with value: 0.85.
[I 2023-02-02 08:53:48,062] Trial 3 finished with value: 0.905 and parameters: {'num_leaves': 1940, 'n_

[I 2023-02-02 08:54:59,170] Trial 30 finished with value: 0.898 and parameters: {'num_leaves': 860, 'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 19, 'reg_alpha': 0.0836165689963788, 'reg_lambda': 0.896652071218075}. Best is trial 3 with value: 0.905.
[I 2023-02-02 08:55:09,820] Trial 31 finished with value: 0.903 and parameters: {'num_leaves': 1460, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 19, 'reg_alpha': 0.009836094835253445, 'reg_lambda': 0.782134092462924}. Best is trial 3 with value: 0.905.
[I 2023-02-02 08:55:11,646] Trial 32 finished with value: 0.903 and parameters: {'num_leaves': 1120, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.11881508513902404, 'reg_lambda': 0.5862732131712031}. Best is trial 3 with value: 0.905.
[I 2023-02-02 08:55:13,501] Trial 33 finished with value: 0.912 and parameters: {'num_leaves': 1380, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.018733892747143946, 'reg_lambda': 

[I 2023-02-02 08:55:44,991] Trial 60 finished with value: 0.892 and parameters: {'num_leaves': 1600, 'n_estimators': 500, 'learning_rate': 0.009, 'max_depth': 5, 'reg_alpha': 0.5892547645414297, 'reg_lambda': 0.9836473721653588}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:55:46,130] Trial 61 finished with value: 0.914 and parameters: {'num_leaves': 1160, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.1261931928964233, 'reg_lambda': 0.8200875095742417}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:55:47,115] Trial 62 finished with value: 0.913 and parameters: {'num_leaves': 980, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.2016151251007037, 'reg_lambda': 0.862769634508777}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:55:48,192] Trial 63 finished with value: 0.91 and parameters: {'num_leaves': 980, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.20797868690546942, 'reg_lambda': 0.

[I 2023-02-02 08:56:26,600] Trial 90 finished with value: 0.911 and parameters: {'num_leaves': 1100, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.3239564808770535, 'reg_lambda': 0.9814571194758871}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:56:27,698] Trial 91 finished with value: 0.911 and parameters: {'num_leaves': 1360, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.1729686257793978, 'reg_lambda': 0.9499152993145856}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:56:29,009] Trial 92 finished with value: 0.91 and parameters: {'num_leaves': 1220, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.11086505577304127, 'reg_lambda': 0.8837128322407434}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:56:29,720] Trial 93 finished with value: 0.905 and parameters: {'num_leaves': 1000, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.5356873257190592, 'reg_lambda': 0

[I 2023-02-02 08:57:14,367] Trial 120 finished with value: 0.847 and parameters: {'num_leaves': 2900, 'n_estimators': 200, 'learning_rate': 0.009, 'max_depth': 19, 'reg_alpha': 0.13327078643301551, 'reg_lambda': 0.9589706057919445}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:57:16,148] Trial 121 finished with value: 0.91 and parameters: {'num_leaves': 1260, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.06660727102224161, 'reg_lambda': 0.9738033036790681}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:57:17,219] Trial 122 finished with value: 0.912 and parameters: {'num_leaves': 880, 'n_estimators': 500, 'learning_rate': 0.3, 'max_depth': 3, 'reg_alpha': 0.1525447199405758, 'reg_lambda': 0.9068677413414147}. Best is trial 54 with value: 0.915.
[I 2023-02-02 08:57:18,907] Trial 123 finished with value: 0.913 and parameters: {'num_leaves': 1120, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.07245428875580144, 'reg_la

...

[I 2023-02-02 09:38:25,756] Trial 900 finished with value: 0.917 and parameters: {'num_leaves': 1020, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.015432057231429775, 'reg_lambda': 0.6633432873619973}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:38:29,342] Trial 901 finished with value: 0.916 and parameters: {'num_leaves': 1080, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.04291580378768675, 'reg_lambda': 0.6428552611579361}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:38:35,174] Trial 902 finished with value: 0.905 and parameters: {'num_leaves': 120, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 9, 'reg_alpha': 0.028184939196129973, 'reg_lambda': 0.6359779641777771}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:38:38,262] Trial 903 finished with value: 0.917 and parameters: {'num_leaves': 880, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.03415643474316496, 'reg

[I 2023-02-02 09:40:16,335] Trial 930 finished with value: 0.918 and parameters: {'num_leaves': 2160, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.019744096563075478, 'reg_lambda': 0.6485432896851467}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:40:19,673] Trial 931 finished with value: 0.917 and parameters: {'num_leaves': 1000, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.029181950905534427, 'reg_lambda': 0.6189236820197579}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:40:20,393] Trial 932 finished with value: 0.911 and parameters: {'num_leaves': 820, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.05108931306489826, 'reg_lambda': 0.6312652179761306}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:40:23,617] Trial 933 finished with value: 0.916 and parameters: {'num_leaves': 380, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.01945923125789289, 'reg

[I 2023-02-02 09:41:53,417] Trial 960 finished with value: 0.917 and parameters: {'num_leaves': 660, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.013447429068063961, 'reg_lambda': 0.7137104170113259}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:41:54,209] Trial 961 finished with value: 0.912 and parameters: {'num_leaves': 960, 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.044569010130979705, 'reg_lambda': 0.6267627794360315}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:41:57,677] Trial 962 finished with value: 0.909 and parameters: {'num_leaves': 1440, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.2784958142265073, 'reg_lambda': 0.6457884812234954}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:42:01,299] Trial 963 finished with value: 0.53 and parameters: {'num_leaves': 2260, 'n_estimators': 500, 'learning_rate': 0.001, 'max_depth': 5, 'reg_alpha': 0.01963052759933786, 're

[I 2023-02-02 09:43:26,715] Trial 990 finished with value: 0.917 and parameters: {'num_leaves': 720, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.03160642197073188, 'reg_lambda': 0.63746896465658}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:43:34,839] Trial 991 finished with value: 0.905 and parameters: {'num_leaves': 1240, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 13, 'reg_alpha': 0.033785375139356894, 'reg_lambda': 0.6361098335320494}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:43:36,457] Trial 992 finished with value: 0.912 and parameters: {'num_leaves': 500, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 3, 'reg_alpha': 0.016102603002652077, 'reg_lambda': 0.6887566581968483}. Best is trial 195 with value: 0.919.
[I 2023-02-02 09:43:39,660] Trial 993 finished with value: 0.917 and parameters: {'num_leaves': 680, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.04735090168204902, 'reg_l

Number of finished trials: 1000


In [15]:
end = time.time()
print(end - start)

3010.921591281891


In [16]:
print('Best trial:', study.best_trial.params)

Best trial: {'num_leaves': 200, 'n_estimators': 500, 'learning_rate': 0.1, 'max_depth': 5, 'reg_alpha': 0.017033744762988044, 'reg_lambda': 0.6995561759520583}


In [17]:
param = {
    "num_leaves": 200,
    "n_estimators": 500,
    'learning_rate': 0.1,
    'max_depth': 5,
    "reg_alpha": 0.017033744762988044,
    'reg_lambda':0.6995561759520583,
    
}


model = LGBMRegressor(**param)

model.fit(train_x,train_y, eval_set=[(test_x, test_y)], verbose=-1)

C:\Users\HuBEL\anaconda3\lib\site-packages\lightgbm\sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMRegressor(max_depth=5, n_estimators=500, num_leaves=200,
              reg_alpha=0.017033744762988044, reg_lambda=0.6995561759520583)

In [18]:
yhat_train = model.predict(train_x)

predicted_train=pd.DataFrame(yhat_train,columns=['Energy'])
predicted_train = inverse_normalization_output(predicted_train)

actual_train = pd.DataFrame(train_y,columns=['Energy'])
actual_train = inverse_normalization_output(actual_train)

print('Training results')
print('r^2:',r2_score(actual_train, predicted_train))
print('RMSE:',np.sqrt(mean_squared_error(actual_train, predicted_train)))
print('MAE:',mean_absolute_error(actual_train, predicted_train))


Training results
r^2: 0.9976263671724687
RMSE: 140.11591967087847
MAE: 97.78784387555393


In [19]:
yhat_test = model.predict(test_x)
predicted_test=pd.DataFrame(yhat_test,columns=['Energy'])
predicted_test = inverse_normalization_output(predicted_test)

actual_test=pd.DataFrame(test_y,columns=['Energy'])
actual_test = inverse_normalization_output(actual_test)

print('Testing results')
print('r^2:',r2_score(actual_test, predicted_test))
print('RMSE:',np.sqrt(mean_squared_error(actual_test, predicted_test)))
print('MAE:',mean_absolute_error(actual_test, predicted_test))

Testing results
r^2: 0.8852689417958254
RMSE: 942.8760476480467
MAE: 520.3248481335913


In [20]:
pickle.dump(model, open('Energy.pkl','wb'))

In [21]:

train_df = pd.concat([actual_train, predicted_train], axis=1)
train_df.to_excel("training_results_past_2.xlsx",index=False)
test_df = pd.concat([actual_test, predicted_test], axis=1)
test_df.to_excel("testing_results_past_2.xlsx",index=False)